# Week 1 Day 2 Lab - Introducing UnderCover (free version)

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a free model, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 3 parts:

PART 1: Examine the data

PART 2: Run with a free model

PART 3: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of the lab. It uses <code>openrouter/free</code> throughout, so it needs no credit on OpenRouter. The full version with paid models is <a href="week1day2.ipynb">week1day2.ipynb</a>.<br/><br/>
            OpenRouter's free models allow 50 requests a day on an account without credit. With <code>TINY = True</code>, the default here, this lab makes about 30. <code>openrouter/free</code> picks a different free model for each request, so expect your results to vary more than mine.<br/><br/>
            Please execute each cell and compare your results with mine. Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [2]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field

## Connect to OpenRouter - and please read note here

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

### NOTE:

If this is a new OpenRouter account, then you might get an error at some point from OpenRouter like: "new accounts are limited to 20 requests per minute for this model"

If you get that error, then add this line immediately after calling an LLM:

`time.sleep(3)`

and you should be in good shape.

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: the Chat Completions API</h2>
            <span style="color:#118AB2;">We call models with the Chat Completions API. <code>messages</code> is a Python list of dicts, each with a <code>role</code> and <code>content</code>. The system prompt tells the model how to behave and the user prompt has the request. The reply is in <code>response.choices[0].message.content</code>.<br/><br/>
            Here's a quick test.
            </span>
        </td>
    </tr>
</table>

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = openrouter.chat.completions.create(model="openrouter/free", messages=messages)
print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by nvidia/nemotron-3-ultra-550b-a55b:free


Ahoy, matey! Ye be askin' the million-doubloon question, the one that’s kept philosophers an' cap’ns starin’ at the horizon till their eyes go square!

**Arr, there be no map with a big red "X" markin' the spot,** no ancient scroll tucked in Davy Jones' locker that says, *"Here lie the Meanin' of It All."*

But if this old salt were to chart a course for yer answer, I’d say the Meanin' be **the voyage itself, not the destination!**

Think on it, ye scurvy dog:

1.  **The Wind in Yer Sails (Freedom):** The universe gave ye a ship—this mortal coil—and a boundless ocean. The meanin’ is hoistin' the colors and sailin' where *ye* choose, not where some landlubber king tells ye.
2.  **The Crew Ye Keep (Mates & Kin):** A captain without a loyal crew is just a madman talkin' to seagulls. The treasure ain't gold, matey—it’s the laughter in the galley, the backs watchin' yours in a squall, the lass or lad waitin' on the shore.
3.  **Weatherin' the Storms (Resilience):** The Kraken *will* rise. The doldrums *will* stall ye. The meanin' is spittin' in the eye o' the tempest, patchin' the hull, and sayin', *"Is that the best ye got?"*
4.  **Leavin' a Wake (Legacy):** When ye finally drop anchor for the last time in Fiddler's Green, did ye leave the seas better than ye found 'em? Did ye sing a shanty loud enough to echo forever?

So, the Meanin' o' the Universe? **It’s whatever ye carve into the mast with yer own cutlass.**

Now, enough philosophizin'! Swab the decks, check the rum ration, and **keep a weather eye on the horizon!** ⚓☠️🗺️

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

The queue has 25 claims. This version sets `TINY = True` to keep 5 of them, which fits the free daily limit. Set it to `False` for all 25 if your account allows more requests.

In [5]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [6]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

5 claims, 5 labels, 50 policies
The policy wording is 1,369 words


In [7]:
claims[0]

{'claim_id': 'CLM-1007',
 'policy_id': 'POL-2007',
 'customer_id': 'CUS-3007',
 'claim_type': 'baggage',
 'submitted_on': '2026-07-22',
 'incident_date': '2026-07-13',
 'incident_location': 'Stockholm, Sweden',
 'summary': 'Phone dropped in the hotel shower',
 'story': 'I dropped my phone in the shower at my hotel in Stockholm on July 13. I know how that sounds. It was on the soap shelf playing music, I was singing along, and during the key change I knocked it off the shelf. It landed screen down in about an inch of water and stayed there longer than it should have, because I had shampoo in my eyes. A repair shop looked at it the next day and said the liquid damage was too extensive to repair. I am claiming the cost of the replacement phone, which was $1,099.',
 'items': [{'description': 'Smartphone, 256 GB, replacement cost',
   'category': 'electronics',
   'amount': 1099.0}],
 'amount_claimed': 1099.0,
 'documents': [{'name': 'Repair shop report',
   'type': 'repair_report',
   'con

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [8]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1007',
 'decision': 'approve',
 'payout': 50.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-07-24',
 'reason': 'Accidental damage during the covered trip, with a repair report stating the phone cannot be repaired. The policy is Economy with no Electronics Plus, so the phone counts at the $150 per-item limit, and the $100 deductible comes off.',
 'reference': 'Policy wording, sections 5 and 10',
 'calculation': [{'step': 'Phone at the per-item limit', 'amount': 150.0},
  {'step': 'Less the deductible', 'amount': -100.0}],
 'tests': 'Payout arithmetic: a per-item limit far below the amount claimed',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'amusing',
 'tiny': True}

### Explore the data

It's always good to dig into the data before running experiments.

In [9]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
1,CLM-1015,Flew home from Paris on arrival after leaving ...,trip_interruption,2340.0,routine,decline,0.0
2,CLM-1017,Backpack snatched by a scooter rider in Naples,baggage,830.0,routine,refer,0.0
3,CLM-1020,Prescription sunglasses stolen by a monkey at ...,baggage,240.0,manual,approve,240.0
4,CLM-1021,Tote bag snatched from her shoulder in Buenos ...,baggage,1103.0,hard,approve,825.0


Claim CLM-1015 might be a familiar example...

In [10]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


Here's another one that might ring a bell...

In [11]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


This chart counts the claims of each type, split by the handler's decision.

In [ ]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [12]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with a free LLM

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [13]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [14]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [ ]:
def decide(claim, model):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
first = decide(claims[0], "openrouter/free")
first

### Now running all the claims

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [ ]:
free_results = run("openrouter/free", claims, labels)

## Results by difficulty

This counts the claims the model got right at each level of difficulty. Expect the manual claims to be wrong for now.

In [ ]:
scored = pd.DataFrame(free_results)
scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum")

## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Pass in another set of results to see its misses.

In [ ]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(free_results, labels)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Accuracy, cost and speed are the numbers a customer will ask about. Cost per claim is the easiest one to relate to the business: compare it with what a handler's time on a claim costs.<br/><br/>
            The kind of mistake matters too. A wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time.
            </span>
        </td>
    </tr>
</table>

## Part 3: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

In [ ]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses `openrouter/free` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with the single calls in Part 2.

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

Now all the claims. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. A claim with no recorded decision counts as wrong.

In [ ]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": "openrouter/free",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
pd.DataFrame(free_results + agent_results).groupby(["model", "technique"], sort=False)[["right", "seconds"]].sum()

In [ ]:
show_misses(agent_results, labels)

## Putting it all together

Here is every experiment in one table. Speed is the average time for one claim.

In [ ]:
rows = []
for results in [free_results, agent_results]:
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand their accuracy and speed.<br/><br/>
            <code>openrouter/free</code> picks a different free model for each request. The free models on OpenRouter each have an id that ends in <code>:free</code>.<br/><br/>
            Challenge: use a specific free model in place of <code>openrouter/free</code>. Which free model gets the most claims right with the agent?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You tried a free model and an agent on UnderCover's claims, with accuracy and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>